# Bronze: Scraping Harga Tokopedia → DuckDB

Notebook ini **berdiri sendiri**, dijalankan terpisah dari notebook GSMArena, tapi **membaca** daftar nama HP
dari `data/gsmarena.duckdb` sebagai kata kunci pencarian, dan menulis hasilnya ke **`data/tokopedia.duckdb`**
(tabel `listings`) — lapisan *bronze* untuk sumber Tokopedia.

```
data/gsmarena.duckdb ──(baca daftar nama HP)──► notebook ini ──► data/tokopedia.duckdb
        (sudah dijalankan lebih dulu)                                   │
                                                                          ▼
                                                        (nanti, notebook build_ducklake
                                                         menggabungkan dgn shopee.duckdb & youtube.duckdb)
```

## Sebelum menjalankan
1. **`data/gsmarena.duckdb` harus sudah berisi data** (jalankan notebook `01_scraping_gsmarena.ipynb` dulu).
2. **Buka Chrome dengan remote debugging** (sama seperti sebelumnya), lalu buka `tokopedia.com` dan lakukan satu
   pencarian manual untuk memastikan tidak ada captcha yang perlu diselesaikan:
```
# Windows (Command Prompt)
"C:\Program Files\Google\Chrome\Application\chrome.exe" --remote-debugging-port=9222 --user-data-dir="C:\chrome_scrape"

# macOS
/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome --remote-debugging-port=9222 --user-data-dir=/tmp/chrome_scrape

# Linux
google-chrome --remote-debugging-port=9222 --user-data-dir=/tmp/chrome_scrape
```

## Checkpoint & lapisan bronze
- Checkpoint berbasis **DuckDB langsung** (tabel `search_progress` mencatat keyword mana yang sudah dicari;
  tabel `listings` menyimpan hasilnya). Jalankan ulang sel scraping kapan saja untuk melanjutkan.
- **Data mentah** (harga masih teks `"Rp1.999.000"`, belum jadi angka) — pembersihan dilakukan nanti di
  `build_ducklake`, sama seperti prinsip di notebook GSMArena.
- **Aksesoris (casing, charger, dll.) disaring saat scraping**, karena itu bukan produk HP sama sekali — sama
  seperti iPad/Watch disaring di notebook GSMArena. Filter harga/kondisi bekas dilakukan nanti di lapisan gold.
- Setelah scraping selesai/berhenti, tabel `listings` **diekspor ke `data/raw_listings_tokopedia.csv`** untuk
  lampiran laporan (murni salinan, bukan sumber data).

## 1. Instalasi & Import Library

In [ ]:
%pip install -q selenium pandas duckdb

In [1]:
import re
import os
import time
import random
from pathlib import Path
from datetime import datetime
from urllib.parse import quote_plus, unquote, urlparse, parse_qs

import pandas as pd
import duckdb
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 2. Konfigurasi

In [ ]:
DATA_DIR = Path("data")
DB_DIR = Path("database")
DATA_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)
GSMARENA_DB = DB_DIR / "gsmarena.duckdb"     # sumber daftar HP (hanya dibaca, tidak ditulis)
DB_PATH = DB_DIR / "tokopedia.duckdb"
CSV_PATH = DATA_DIR / "raw_dataset_tokopedia.csv"
print("Membaca daftar HP dari:", GSMARENA_DB.resolve())
print("Menulis ke:", DB_PATH.resolve())

GITHUB_WARN_MB = 50

DEBUG_PORT = 9222                # port Chrome remote debugging (lihat panduan di atas)
MAX_KEYWORDS = 300                 # jumlah HP (dari gsmarena.duckdb) yang dicari; > N_PHONES_COMPARE nanti di build_ducklake (beri slack)
MAX_PRODUCTS_PER_KEYWORD = 10
TARGET_ROWS = 1800                # berhenti jika listing tersimpan sudah mencapai angka ini (batas wajar per sesi)
MAX_CONSECUTIVE_EMPTY = 12        # berhenti jika sekian keyword berturut-turut kosong (indikasi diblokir/selector berubah)
DELAY = (3.0, 6.0)

KEYWORD_EXCLUDE = ["pad", "tab", "fold", "flip", "watch"]   # jaga-jaga jika ada yang lolos dari filter GSMArena
MIN_PHONE_PRICE = 300_000
EXCLUDE_WORDS = ["case", "casing", "softcase", "tempered", "anti gores", "antigores", "charger",
                 "kabel", "cable", "holder", "sticker", "skin", "pouch", "sarung", "lcd", "sparepart",
                 "baterai", "battery", "earphone", "headset", "powerbank", "dummy", "replika", "mockup"]


def clean_price(text):
    if text is None:
        return None
    m = re.search(r"\d[\d\.,]*", str(text))
    if not m:
        return None
    digits = re.sub(r"[^\d]", "", m.group(0))
    return int(digits) if digits else None


def looks_like_accessory(product_name):
    low = (product_name or "").lower()
    return any(w in low for w in EXCLUDE_WORDS)


def save_csv():
    df = con.execute("SELECT * FROM listings ORDER BY keyword, scraped_at").df()
    try:
        df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    except PermissionError:
        alt = DB_DIR / f"{datetime.now():%Y%m%d_%H%M%S}_{CSV_PATH.name}"
        df.to_csv(alt, index=False, encoding="utf-8-sig")
        print(f"⚠️ File asli terkunci, disimpan sebagai: {alt}")
        return
    print(f"✔ {CSV_PATH} diperbarui ({len(df)} baris)")

Membaca daftar HP dari: C:\KULIAH\Semester 7\Data Mining II\Tugas\PJBL\data\gsmarena.duckdb
Menulis ke: C:\KULIAH\Semester 7\Data Mining II\Tugas\PJBL\data\tokopedia.duckdb


## 3. Baca Daftar HP dari `gsmarena.duckdb`
Keyword dipilih **bergantian antar brand** (round-robin), dimulai dari HP yang lebih baru, supaya semua brand
terwakili sejak awal (bukan menghabiskan kuota di satu brand dulu).

In [3]:
if not GSMARENA_DB.exists():
    raise RuntimeError(f"{GSMARENA_DB} tidak ditemukan. Jalankan notebook 01_scraping_gsmarena.ipynb terlebih dahulu.")

gsm = duckdb.connect(str(GSMARENA_DB), read_only=True)
specs = gsm.execute("SELECT brand, phone_name, release_year FROM phone_specs").df()
gsm.close()
print(f"{len(specs)} HP tersedia di gsmarena.duckdb")

ok = ~specs["phone_name"].str.lower().str.split().apply(lambda ws: bool(set(ws) & set(KEYWORD_EXCLUDE)))
groups = [g.sort_values("release_year", ascending=False, na_position="last")["phone_name"].tolist()
          for _, g in specs[ok].groupby("brand")]
keywords = []
for i in range(max((len(g) for g in groups), default=0)):
    for g in groups:
        if i < len(g):
            keywords.append(g[i])
keywords = keywords[:MAX_KEYWORDS]

print(f"{len(keywords)} keyword siap dicari. Contoh: {keywords[:8]}")

282 HP tersedia di gsmarena.duckdb
281 keyword siap dicari. Contoh: ['Apple iPhone Duo', 'Huawei Mate XT 2 Ultimate', 'Infinix Xpad 30 Pro', 'Oppo Find X10 Pro Max', 'Realme C100i (India)', 'Samsung Galaxy A18 4G', 'Tecno Camon Air', 'vivo iQOO 16']


## 4. Koneksi ke `data/tokopedia.duckdb`
Dua tabel: `listings` (hasil scraping mentah) dan `search_progress` (checkpoint — keyword mana yang sudah dicari).

In [4]:
con = duckdb.connect(str(DB_PATH))
con.execute("""CREATE TABLE IF NOT EXISTS listings (
    keyword VARCHAR, product_name VARCHAR, product_url VARCHAR, price_discount_raw VARCHAR,
    price_original_raw VARCHAR, discount_label_raw VARCHAR, rating_raw VARCHAR, sold_raw VARCHAR,
    seller_name VARCHAR, seller_location VARCHAR, scraped_at TIMESTAMP
)""")
con.execute("""CREATE TABLE IF NOT EXISTS search_progress (
    keyword VARCHAR PRIMARY KEY, n_found INTEGER, searched_at TIMESTAMP
)""")

done_keywords = set(con.execute("SELECT keyword FROM search_progress").df()["keyword"])
n_rows = con.execute("SELECT count(*) FROM listings").fetchone()[0]
print(f"Checkpoint: {len(done_keywords)} keyword sudah dicari, {n_rows} listing tersimpan.")

Checkpoint: 74 keyword sudah dicari, 317 listing tersimpan.


## 5. Extract: Tokopedia (Selenium + BeautifulSoup)
Selenium **menyambung ke Chrome yang sudah Anda buka manual** (bukan membuka browser baru), supaya tidak
terdeteksi sebagai otomasi dan captcha bisa Anda selesaikan sendiri.
- **Parser utama** memakai `data-testid`; jika selector itu berubah, **parser cadangan** mengambil semua link
  yang teksnya memuat `"Rp"` dan mengekstrak info dengan regex.
- **Halaman blokir** ("Verifikasi gagal", dsb.) membuat proses **berhenti**, bukan dipaksa terus, karena
  mengulang-ulang saat diblokir hanya memperpanjang blokirnya.

In [5]:
BLOCK_PATTERNS = ["silakan coba lagi nanti", "verifikasi gagal", "coba lagi beberapa menit",
                  "too many requests", "access denied"]


def make_driver():
    opts = Options()
    opts.add_experimental_option("debuggerAddress", f"127.0.0.1:{DEBUG_PORT}")
    return webdriver.Chrome(options=opts)


def polite_sleep(a, b):
    time.sleep(random.uniform(a, b))


def scroll_page(driver, times=8):
    for _ in range(times):
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.8);")
        time.sleep(1.0)


def is_blocked(html):
    low = html.lower()
    return any(p in low for p in BLOCK_PATTERNS) and 'data-testid="master-product-card"' not in html


def needs_human(driver):
    url = driver.current_url.lower()
    if any(w in url for w in ["login", "verify", "captcha", "punish", "security"]):
        return True
    return is_blocked(driver.page_source)


def first_text(card, selectors):
    for sel in selectors:
        el = card.select_one(sel)
        if el and el.get_text(strip=True):
            return el.get_text(strip=True)
    return None


def extract_sold(tail):
    m = (re.search(r"terjual\s*([\d\.,]+\s*(?:rb|jt)?\+?)", tail, re.I)
         or re.search(r"([\d\.,]+\s*(?:rb|jt)?\+?)\s*terjual", tail, re.I))
    return m.group(1).strip() if m else None


def shop_from_url(href):
    if "ta.tokopedia.com" in href:
        r = parse_qs(urlparse(href).query).get("r")
        if r:
            href = unquote(r[0])
    parts = [p for p in urlparse(href).path.split("/") if p]
    return parts[0] if len(parts) >= 2 else None


def parse_testid(html, keyword, limit):
    soup = BeautifulSoup(html, "lxml")
    rows = []
    for card in soup.select('[data-testid="master-product-card"]')[:limit]:
        link = card.select_one("a[href]")
        rows.append({
            "keyword": keyword,
            "product_name": first_text(card, ['[data-testid="spnSRPProdName"]']),
            "product_url": (link["href"].split("?")[0] if link else None),
            "price_discount_raw": first_text(card, ['[data-testid="spnSRPProdPrice"]']),
            "price_original_raw": first_text(card, ['[data-testid="lblProductSlashPrice"]']),
            "discount_label_raw": first_text(card, ['[data-testid="lblProductDiscount"]']),
            "rating_raw": first_text(card, ['span.prd_rating-average-text']),
            "sold_raw": first_text(card, ['span.prd_label-integrity']),
            "seller_name": first_text(card, ['span.prd_link-shop-name', '[data-testid="spnSRPProdTabShopName"]']),
            "seller_location": first_text(card, ['span.prd_link-shop-loc', '[data-testid="spnSRPProdTabShopLoc"]']),
        })
    return rows


def parse_generic(html, keyword, limit):
    soup = BeautifulSoup(html, "lxml")
    rows, seen = [], set()
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True)
        href = a["href"]
        if "Rp" not in text or "tokopedia.com" not in href or href in seen:
            continue
        price_matches = list(re.finditer(r"Rp\s?[\d\.]+", text))
        nums = sorted(p for p in (clean_price(m.group(0)) for m in price_matches) if p)
        if not nums:
            continue
        seen.add(href)
        tail = text[price_matches[-1].end():]
        rating = re.search(r"\b([1-5][.,]\d)\b", tail)
        rows.append({
            "keyword": keyword, "product_name": text.split("Rp")[0].strip(" -|"), "product_url": href.split("?")[0],
            "price_discount_raw": f"Rp{nums[0]}", "price_original_raw": f"Rp{nums[-1]}" if len(nums) > 1 else None,
            "discount_label_raw": None, "rating_raw": rating.group(1) if rating else None,
            "sold_raw": extract_sold(tail), "seller_name": shop_from_url(href), "seller_location": None,
        })
        if len(rows) >= limit:
            break
    return rows

In [6]:
def scrape_tokopedia():
    driver = make_driver()
    empty_streak = 0
    total = con.execute("SELECT count(*) FROM listings").fetchone()[0]

    for kw in keywords:
        if total >= TARGET_ROWS:
            print(f"✅ Target {TARGET_ROWS} baris tercapai.")
            break
        if kw in done_keywords:
            continue

        print(f"({total}/{TARGET_ROWS}) Cari: {kw}")
        driver.get(f"https://www.tokopedia.com/search?q={quote_plus(kw)}&st=product")
        time.sleep(3)

        if is_blocked(driver.page_source):
            print("⛔ Tokopedia menampilkan halaman blokir/verifikasi. Proses dihentikan agar tidak memperburuk.\n"
                  "   Tunggu ± 15-30 menit, atau selesaikan verifikasi manual di Chrome, lalu jalankan ulang sel ini.")
            break
        if needs_human(driver):
            input("   Login/captcha terdeteksi. Selesaikan di jendela Chrome, lalu tekan Enter di sini...")
            driver.get(f"https://www.tokopedia.com/search?q={quote_plus(kw)}&st=product")

        try:
            WebDriverWait(driver, 15).until(lambda d: len(d.find_elements(By.XPATH, "//a[contains(., 'Rp')]")) > 0)
            scroll_page(driver)
            html = driver.page_source
        except TimeoutException:
            html = ""

        found = parse_testid(html, kw, MAX_PRODUCTS_PER_KEYWORD) if html else []
        if not found and html:
            found = parse_generic(html, kw, MAX_PRODUCTS_PER_KEYWORD)
        found = [r for r in found if not looks_like_accessory(r["product_name"])]   # buang aksesoris (bukan HP)
        print(f"   {len(found)} listing (setelah buang aksesoris)")

        if found:
            rows = [{**r, "scraped_at": None} for r in found]
            con.executemany(
                "INSERT INTO listings VALUES (?,?,?,?,?,?,?,?,?,?, now())",
                [[r["keyword"], r["product_name"], r["product_url"], r["price_discount_raw"], r["price_original_raw"],
                  r["discount_label_raw"], r["rating_raw"], r["sold_raw"], r["seller_name"], r["seller_location"]]
                 for r in rows])
            total += len(found)
            empty_streak = 0
        else:
            empty_streak += 1

        con.execute("INSERT OR REPLACE INTO search_progress VALUES (?,?, now())", [kw, len(found)])
        done_keywords.add(kw)

        if empty_streak >= MAX_CONSECUTIVE_EMPTY:
            print(f"⛔ {MAX_CONSECUTIVE_EMPTY} keyword berturut-turut kosong (kemungkinan diblokir/selector berubah). "
                  "Jalankan diagnose() lalu ulangi (proses akan melanjutkan).")
            break
        polite_sleep(*DELAY)

    save_csv()
    size_mb = DB_PATH.stat().st_size / 1_000_000
    flag = " ⚠️ mendekati/melebihi batas GitHub!" if size_mb > GITHUB_WARN_MB else ""
    print(f"\nUkuran {DB_PATH.name}: {size_mb:.1f} MB{flag}")
    # driver.quit() sengaja tidak dipanggil agar jendela Chrome Anda tetap terbuka


def diagnose(keyword="samsung galaxy a55"):
    driver = make_driver()
    driver.get(f"https://www.tokopedia.com/search?q={quote_plus(keyword)}&st=product")
    time.sleep(8)
    scroll_page(driver, 3)
    html = driver.page_source
    print("Judul halaman :", driver.title)
    print("URL sekarang  :", driver.current_url)
    print("Halaman diblokir?:", is_blocked(html))
    print("Butuh login/captcha?:", needs_human(driver))
    print("data-testid yang ada:", sorted(set(re.findall(r'data-testid="([^"]+)"', html)))[:60])
    print("Hasil parser utama:", len(parse_testid(html, keyword, 50)), "| parser cadangan:", len(parse_generic(html, keyword, 50)))
    driver.save_screenshot(str(DB_DIR / "debug_tokopedia.png"))
    (DB_DIR / "debug_tokopedia.html").write_text(html, encoding="utf-8")
    print(f"Screenshot & HTML disimpan di {DB_DIR}/debug_tokopedia.*")

## 6. Jalankan Scraping
Bisa dihentikan & dilanjutkan kapan saja.

In [8]:
scrape_tokopedia()

n = con.execute("SELECT count(*) FROM listings").fetchone()[0]
if n == 0:
    raise RuntimeError("Belum ada listing tersimpan. Jalankan diagnose() untuk memeriksa penyebabnya.")
print(f"\nTotal listing tersimpan: {n}")

(553/1800) Cari: Realme C100 4G
   9 listing (setelah buang aksesoris)
(562/1800) Cari: Samsung Galaxy A27
   10 listing (setelah buang aksesoris)
(572/1800) Cari: Tecno Spark 50
   8 listing (setelah buang aksesoris)
(580/1800) Cari: vivo Y31t
   7 listing (setelah buang aksesoris)
(587/1800) Cari: Xiaomi Redmi Note 17 4G
   0 listing (setelah buang aksesoris)
(587/1800) Cari: itel VistaTab 10 mini
   0 listing (setelah buang aksesoris)
(587/1800) Cari: Apple iPhone 16 Plus
   10 listing (setelah buang aksesoris)
(597/1800) Cari: Huawei nova 16
   10 listing (setelah buang aksesoris)
(607/1800) Cari: Infinix GT 30
   10 listing (setelah buang aksesoris)
(617/1800) Cari: Oppo Reno16 FS
   9 listing (setelah buang aksesoris)
(626/1800) Cari: Realme C100
   7 listing (setelah buang aksesoris)
(633/1800) Cari: Samsung Galaxy A57
   10 listing (setelah buang aksesoris)
(643/1800) Cari: Tecno Camon 50 Ultra
   0 listing (setelah buang aksesoris)
(643/1800) Cari: vivo iQOO Neo11 Ultra
   7 l

### Diagnosa (jalankan hanya jika hasil scraping kosong)
Hilangkan tanda `#`, jalankan, lalu baca outputnya.

In [ ]:
# diagnose()

## 7. Pratinjau Data

In [9]:
con.execute("""
    SELECT keyword, count(*) AS jumlah_listing
    FROM listings GROUP BY keyword ORDER BY jumlah_listing DESC LIMIT 15
""").df()

,keyword,jumlah_listing
0,Oppo Reno16 (China),10
1,Xiaomi Redmi Note 17 (India/China),10
2,Infinix Note 50 Pro 4G,10
3,itel A90,10
4,Tecno Spark 40 Pro+,10
5,Samsung Galaxy F07,10
6,Realme 16 Pro+,10
7,Apple iPhone 17e,10
8,Infinix GT 30,10
9,itel Super 26 Ultra,10


In [10]:
con.execute("SELECT * FROM listings ORDER BY scraped_at DESC LIMIT 10").df()

,keyword,product_name,product_url,price_discount_raw,price_original_raw,discount_label_raw,rating_raw,sold_raw,seller_name,seller_location,scraped_at
0,vivo iQOO Pad6 Pro,iQOO Z9 5G Android Snapdragon7 Gen3 smartphone...,https://www.tokopedia.com/tokosrcfedanes/iqoo-...,Rp8581000,Rp8606000,None,NaN,NaN,tokosrcfedanes,None,2026-10-03 10:26:41.430091
1,vivo iQOO Pad6 Pro,34% Xprime S25 10.1 Inch 120Hz 16GB+1TB 8800mA...,https://www.tokopedia.com/ax-prime/xprime-s25-...,Rp1765400,Rp2688000,None,4.9,250+,ax-prime,None,2026-10-03 10:26:41.427688
2,vivo iQOO Pad6 Pro,( Special Creator ) Samsung Galaxy Tab S10 Fe ...,https://www.tokopedia.com/ini-toko-budi-shop/s...,Rp8749000,Rp8750000,None,4.9,1rb+,ini-toko-budi-shop,None,2026-10-03 10:26:41.424688
3,vivo iQOO Pad6 Pro,Xiaomi Pad 8 & Pad 8 Pro Chipset Snapdragon 8 ...,https://www.tokopedia.com/doran-gadget-authori...,Rp12029000,NaN,None,5.0,4,doran-gadget-authorized-shop,None,2026-10-03 10:26:41.418320
4,Tecno Pova 7 Ultra,Mall [Andriyadishow] TECNO POVA 7 Ultra 5G 12+...,https://www.tokopedia.com/xymgicor08/mall-andr...,Rp1175000,Rp1200000,None,NaN,1,xymgicor08,None,2026-10-03 10:26:06.881579
5,Tecno Pova 7 Ultra,"Promo Tecno POVA 7 Ultra 5G 12/256GB 6.67"" AMO...",https://www.tokopedia.com/toko-septisari77/pro...,Rp4479000,Rp4504000,None,NaN,1,toko-septisari77,None,2026-10-03 10:26:06.878959
6,Tecno Pova 7 Ultra,TECNO POVA 7 ULTRA - 12/256GB RAM - DIMENSITY ...,https://www.tokopedia.com/impostore/tecno-pova...,Rp4875000,Rp4900000,None,NaN,2,impostore,None,2026-10-03 10:26:06.876195
7,Tecno Pova 7 Ultra,Tecno pova 7 ultra ram 12 / 256 resmi bnib seg...,https://www.tokopedia.com/f44i-shop/tecno-pova...,Rp5374900,Rp5399900,None,NaN,1,f44i-shop,None,2026-10-03 10:26:06.873469
8,Tecno Pova 7 Ultra,Tecno pova 7 ultra ram 12 / 256 resmi unit only,https://www.tokopedia.com/f44i-shop/tecno-pova...,Rp3825000,Rp3850000,None,5.0,1,f44i-shop,None,2026-10-03 10:26:06.870843
9,Tecno Pova 7 Ultra,POVA 7 ULTRA 5G 12/256GB SEGEL ORIGINAL & BERG...,https://www.tokopedia.com/gadgedmurah-1/pova-7...,Rp5300000,NaN,None,NaN,1,gadgedmurah-1,None,2026-10-03 10:26:06.868286


## Catatan
- **Lapisan bronze, belum dibersihkan.** Harga masih teks (`"Rp1.999.000"`), belum dijadikan angka. Pembersihan,
  pencocokan nama HP ke `dim_phone`, dan penyamaan jumlah data dengan Shopee dilakukan di notebook `build_ducklake`.
- **`data/raw_listings_tokopedia.csv`** hanya salinan untuk laporan — selalu dibuat ulang dari isi tabel DuckDB,
  jadi aman meski sel scraping dijalankan berkali-kali.
- **Aksesoris disaring saat scraping** (bukan produk HP sama sekali, sama prinsipnya dengan iPad/Watch di GSMArena).
  Filter harga minimum, HP bekas, dan pencocokan nama tetap dilakukan di lapisan gold, karena itu keputusan
  analisis, bukan soal jenis produk yang salah.